# `locityper_00_run_stream` — Locityper on Verily Workbench

Run **in a VWB JupyterLab app** (`wb` on `PATH`). This is the CLI path around
the broken Workflows GUI.

Isaac’s Terra workflow:
[`locityper_stream.wdl`](https://github.com/EichlerLab/AoU_WDL/blob/main/locityper/locityper_stream.wdl)
(`ValidateVariants`). The copy in this repo is `LocityperStream`
([`locityper/README.md`](../../locityper/README.md)). Submission follows Matt’s
`eTRs_getPhasedAlleleInfo` notebook: stage a WDL to a workspace bucket, then
`wb workflow job run`.

Docs: [Cromwell in Workbench](https://support.workbench.verily.com/docs/guides/workflows/cromwell/),
[`wb workflow job run`](https://support.workbench.verily.com/docs/references/cli_reference/wb/workflow/job/run/).

## What this notebook does

1. Check `wb` auth / workspace
2. Copy `LocityperStream.wdl` into a workspace GCS bucket
3. `wb workflow create` (`locityper-stream`) if it is missing
4. Write and upload a **one-sample** inputs JSON (smoke test)
5. Optionally submit that job
6. Optionally build a batch CSV and submit one job per row

`SUBMIT_SINGLE` and `SUBMIT_BATCH` stay **off** until you turn them on.
Fill every `gs://` path in the config cell first.

The genotype image (`eichlerlab/locityper:1.4.5.0`) must be pullable from
Cromwell workers. VPC-SC workspaces usually need an Artifact Registry mirror;
set `LOCITYPER_DOCKER` / `UTIL_DOCKER` in that case.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        wdl = p / "locityper" / "wdl" / "LocityperStream.wdl"
        if wdl.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "locityper" / "wdl" / "LocityperStream.wdl"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find locityper/wdl/LocityperStream.wdl. Clone kvg/aou-lr-phase-2 "
        "into this app (or cd into the clone) and re-run."
    )


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


def capture(cmd: list[str], *, check: bool = True) -> str:
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, check=check, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n", file=sys.stderr)
    return proc.stdout


def wb_json(*args: str) -> Any:
    raw = capture(["wb", *args, "--format=JSON"])
    raw = raw.strip()
    if not raw:
        return None
    return json.loads(raw)


def gcs_cp(src: str | Path, dest: str) -> None:
    sh(["gcloud", "storage", "cp", str(src), dest])


REPO_ROOT = find_repo_root()
WDL_LOCAL = REPO_ROOT / "locityper" / "wdl" / "LocityperStream.wdl"
EXAMPLE_INPUTS = REPO_ROOT / "locityper" / "configs" / "locityper.inputs.json.example"
COLUMN_MAPPING_LOCAL = REPO_ROOT / "locityper" / "configs" / "column_mapping.json"
SCRATCH = Path.cwd() / "locityper_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

# Workspace resource ID (not the gs:// name). `wb resource list --type=GCS_BUCKET`.
OUTPUT_BUCKET_ID = os.environ.get("LOCITYPER_OUTPUT_BUCKET_ID", "")
# Optional override if resolve is wrong; otherwise filled after listing buckets.
OUTPUT_BUCKET_GS = os.environ.get("LOCITYPER_OUTPUT_BUCKET_GS", "")

WORKFLOW_ID = os.environ.get("LOCITYPER_WORKFLOW_ID", "locityper-stream")
WDL_BUCKET_PATH = os.environ.get("LOCITYPER_WDL_PATH", "wdl/locityper/LocityperStream.wdl")
OUTPUT_PATH = os.environ.get("LOCITYPER_OUTPUT_PATH", "workflowRuns/locityper")
STAGE_PREFIX = os.environ.get("LOCITYPER_STAGE_PREFIX", "locityper")

# Isaac's Terra defaults (small CPU). Bump locityper_n_cpu for production.
SAMPLE_ID = os.environ.get("LOCITYPER_SAMPLE_ID", "")
CRAM = os.environ.get("LOCITYPER_CRAM", "")
CRAI = os.environ.get("LOCITYPER_CRAI", "")
REF_FA = os.environ.get("LOCITYPER_REF_FA", "")
REF_FAI = os.environ.get("LOCITYPER_REF_FAI", "")
COUNTS_JF = os.environ.get("LOCITYPER_COUNTS_JF", "")
BED = os.environ.get("LOCITYPER_BED", "")
DB_TAR = os.environ.get("LOCITYPER_DB_TAR", "")

TECHNOLOGY = os.environ.get("LOCITYPER_TECHNOLOGY", "illumina")
N_SPLIT = int(os.environ.get("LOCITYPER_N_SPLIT", "150"))
LOCITYPER_N_CPU = int(os.environ.get("LOCITYPER_N_CPU", "2"))
LOCITYPER_EXTRA_MEM_GB = int(os.environ.get("LOCITYPER_EXTRA_MEM_GB", "4"))
WINDOW_GRAB = int(os.environ.get("LOCITYPER_WINDOW_GRAB", "3000"))
MAX_RETRY = int(os.environ.get("LOCITYPER_MAX_RETRY", "7"))
WAIT_TIME = int(os.environ.get("LOCITYPER_WAIT_TIME", "75"))
N_PREEMPTIBLE = int(os.environ.get("LOCITYPER_N_PREEMPTIBLE", "2"))

LOCITYPER_DOCKER = os.environ.get("LOCITYPER_DOCKER", "eichlerlab/locityper:1.4.5.0")
UTIL_DOCKER = os.environ.get("LOCITYPER_UTIL_DOCKER", "python:3.11-slim")

# Optional path to a local batch CSV (same columns as
# locityper/configs/batch.header.csv). If empty, the batch cell writes a
# one-row CSV from SAMPLE_ID / CRAM / … in this cell.
BATCH_CSV_LOCAL = os.environ.get("LOCITYPER_BATCH_CSV", "")

STAGE_WDL = True
REGISTER_WORKFLOW = True
RECREATE_WORKFLOW = False
SUBMIT_SINGLE = False
SUBMIT_BATCH = False
CANCEL_JOB_ID = ""  # set to a job UUID to cancel in the last cell

print("REPO_ROOT:", REPO_ROOT)
print("WDL_LOCAL:", WDL_LOCAL, "exists=" + str(WDL_LOCAL.is_file()))
print("OUTPUT_BUCKET_ID:", OUTPUT_BUCKET_ID or "(unset)")
print("WORKFLOW_ID:", WORKFLOW_ID)
print("SAMPLE_ID:", SAMPLE_ID or "(unset)")
print("SUBMIT_SINGLE:", SUBMIT_SINGLE, "SUBMIT_BATCH:", SUBMIT_BATCH)
print("wb:", shutil.which("wb"))


## Workspace and buckets

`--output-bucket-id` is the **resource ID** from `wb resource list`, not the
`gs://` name. Matt’s eTR notebook used `rw-migration-aou-rw-0b461bba` for a
bucket whose cloud path was `gs://cloned-rw-migration-…`.


In [ ]:
if shutil.which("wb") is None:
    raise SystemExit("wb is not on PATH. Open this notebook in a Verily Workbench Jupyter app.")

sh(["wb", "version"], check=False)
sh(["wb", "auth", "status"], check=False)
sh(["wb", "status"], check=False)

print("\n--- GCS buckets in this workspace ---")
sh(["wb", "resource", "list", "--type=GCS_BUCKET"], check=False)

bucket_json = None
try:
    bucket_json = wb_json("resource", "list", "--type=GCS_BUCKET")
except Exception as exc:  # noqa: BLE001
    print("could not parse JSON bucket list:", exc)

if not OUTPUT_BUCKET_ID:
    print(
        "\nSet OUTPUT_BUCKET_ID in the config cell to a resource ID from the list above."
    )
else:
    resolved = capture(["wb", "resource", "resolve", f"--id={OUTPUT_BUCKET_ID}"], check=False).strip().splitlines()
    resolved = next((line.strip() for line in reversed(resolved) if line.strip()), "")
    if resolved and not OUTPUT_BUCKET_GS:
        if resolved.startswith("gs://") or resolved.startswith("s3://"):
            OUTPUT_BUCKET_GS = resolved
        elif " " not in resolved and "/" not in resolved.split(":")[0]:
            OUTPUT_BUCKET_GS = f"gs://{resolved}"
        else:
            print("could not parse resolve output; set OUTPUT_BUCKET_GS in the config cell")
    print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS or "(unset)")


## Stage the WDL and register `locityper-stream`

Workbench cannot register a Git path as a workflow. Copy the WDL into the
workspace bucket, then `wb workflow create`. Re-run with `RECREATE_WORKFLOW=True`
after you change the WDL (that deletes the workspace workflow resource, not
GCS outputs).


In [ ]:
if not OUTPUT_BUCKET_ID or not OUTPUT_BUCKET_GS:
    raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before staging.")

WDL_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{WDL_BUCKET_PATH}"
INPUTS_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/inputs.smoke.json"
COLUMN_MAPPING_URI = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX}/column_mapping.json"
BATCH_CSV_URI_PATH = f"{STAGE_PREFIX}/batch.csv"

if STAGE_WDL:
    gcs_cp(WDL_LOCAL, WDL_URI)
    print("staged WDL:", WDL_URI)
else:
    print("STAGE_WDL=False; skip copy")

print("\n--- workflows already in the workspace ---")
sh(["wb", "workflow", "list"], check=False)

existing_ids: set[str] = set()
try:
    listed = wb_json("workflow", "list")
    rows = listed if isinstance(listed, list) else (listed or {}).get("result") or (listed or {}).get("workflows") or []
    if isinstance(rows, dict):
        rows = [rows]
    for row in rows:
        if not isinstance(row, dict):
            continue
        for key in ("id", "workflowId", "workflow_id", "resourceId"):
            if row.get(key):
                existing_ids.add(str(row[key]))
except Exception as exc:  # noqa: BLE001
    print("could not parse workflow list JSON:", exc)

registered = WORKFLOW_ID in existing_ids
print(f"{WORKFLOW_ID} already registered:", registered)

if RECREATE_WORKFLOW and registered:
    sh(["wb", "workflow", "delete", "--quiet", f"--workflow={WORKFLOW_ID}"])
    registered = False

if REGISTER_WORKFLOW and not registered:
    sh(
        [
            "wb",
            "workflow",
            "create",
            f"--bucket-id={OUTPUT_BUCKET_ID}",
            f"--path={WDL_BUCKET_PATH}",
            f"--workflow={WORKFLOW_ID}",
            "--workflow-type=WDL",
            "--display-name=Locityper stream",
            "--description=Isaac locityper_stream (LocityperStream) for VWB CLI submit",
        ]
    )
elif registered:
    print(f"leave existing workflow {WORKFLOW_ID} in place")

sh(["wb", "workflow", "describe", f"--workflow={WORKFLOW_ID}"], check=False)


## Smoke-test inputs (one sample)

Required: `SAMPLE_ID`, `CRAM`, `CRAI`, uncompressed reference + fai, jellyfish
counts, 4-column loci BED, `vcf_db.tar.gz`. File values must be `gs://` URIs
the PET SA can read.

Isaac’s Terra JSON used `N_split=150`, `locityper_n_cpu=2`, `window_grab=3000`,
`technology=illumina`. Use a **tiny BED** and one CRAM before a cohort.


In [ ]:
required = {
    "SAMPLE_ID": SAMPLE_ID,
    "CRAM": CRAM,
    "CRAI": CRAI,
    "REF_FA": REF_FA,
    "REF_FAI": REF_FAI,
    "COUNTS_JF": COUNTS_JF,
    "BED": BED,
    "DB_TAR": DB_TAR,
}
missing = [k for k, v in required.items() if not v]
if missing:
    print("Not uploading yet; set these in the config cell:", ", ".join(missing))
    INPUTS = None
else:
    INPUTS = {
        "LocityperStream.sample_id": SAMPLE_ID,
        "LocityperStream.cram": CRAM,
        "LocityperStream.crai": CRAI,
        "LocityperStream.ref_fa_uncompressed": REF_FA,
        "LocityperStream.ref_fai_uncompressed": REF_FAI,
        "LocityperStream.counts_jf": COUNTS_JF,
        "LocityperStream.bed": BED,
        "LocityperStream.locityper_db_tar_gz": DB_TAR,
        "LocityperStream.N_split": N_SPLIT,
        "LocityperStream.locityper_n_cpu": LOCITYPER_N_CPU,
        "LocityperStream.locityper_extra_mem_gb": LOCITYPER_EXTRA_MEM_GB,
        "LocityperStream.window_grab": WINDOW_GRAB,
        "LocityperStream.max_retry": MAX_RETRY,
        "LocityperStream.wait_time": WAIT_TIME,
        "LocityperStream.n_preemptible": N_PREEMPTIBLE,
        "LocityperStream.technology": TECHNOLOGY,
        "LocityperStream.locityper_docker": LOCITYPER_DOCKER,
        "LocityperStream.util_docker": UTIL_DOCKER,
    }
    inputs_path = SCRATCH / "inputs.smoke.json"
    inputs_path.write_text(json.dumps(INPUTS, indent=2) + "\n")
    print(inputs_path.read_text())
    gcs_cp(inputs_path, INPUTS_URI)
    print("uploaded", INPUTS_URI)


## Submit one job

Turn on `SUBMIT_SINGLE` in the config cell after the inputs upload succeeds.


In [ ]:
if not SUBMIT_SINGLE:
    print("SUBMIT_SINGLE=False; not submitting. Set True in the config cell and re-run.")
elif INPUTS is None:
    raise SystemExit("Fill SAMPLE_ID / CRAM / … before submitting.")
else:
    sh(
        [
            "wb",
            "workflow",
            "job",
            "run",
            f"--workflow={WORKFLOW_ID}",
            f"--output-bucket-id={OUTPUT_BUCKET_ID}",
            f"--output-path={OUTPUT_PATH}",
            f"--inputs-uri={INPUTS_URI}",
            "--delete-intermediate-outputs",
        ]
    )


## Optional: batch CSV

Workbench batch jobs take a CSV in a workspace bucket plus a column map.
Shared files (`bed`, reference, DB) must be **repeated on every row**.

If `BATCH_CSV_LOCAL` points at a file, that file is uploaded as-is (after an
optional row-0 prepend of the smoke-test sample). Otherwise this cell writes a
one-row CSV from the config cell.


In [ ]:
BATCH_FIELDS = [
    "sample_id",
    "cram",
    "crai",
    "ref_fa_uncompressed",
    "ref_fai_uncompressed",
    "counts_jf",
    "bed",
    "locityper_db_tar_gz",
]

batch_path = SCRATCH / "batch.csv"
if BATCH_CSV_LOCAL:
    src = Path(BATCH_CSV_LOCAL)
    if not src.is_file():
        raise FileNotFoundError(src)
    shutil.copyfile(src, batch_path)
else:
    if missing:
        print("No batch CSV: config sample fields still missing:", ", ".join(missing))
        batch_path = None
    else:
        with batch_path.open("w", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=BATCH_FIELDS)
            writer.writeheader()
            writer.writerow(
                {
                    "sample_id": SAMPLE_ID,
                    "cram": CRAM,
                    "crai": CRAI,
                    "ref_fa_uncompressed": REF_FA,
                    "ref_fai_uncompressed": REF_FAI,
                    "counts_jf": COUNTS_JF,
                    "bed": BED,
                    "locityper_db_tar_gz": DB_TAR,
                }
            )

if batch_path is not None:
    print(batch_path.read_text())
    gcs_cp(batch_path, f"{OUTPUT_BUCKET_GS.rstrip('/')}/{BATCH_CSV_URI_PATH}")
    gcs_cp(COLUMN_MAPPING_LOCAL, COLUMN_MAPPING_URI)
    print("batch CSV:", f"{OUTPUT_BUCKET_GS.rstrip('/')}/{BATCH_CSV_URI_PATH}")
    print("column map:", COLUMN_MAPPING_URI)

    if not SUBMIT_BATCH:
        print("SUBMIT_BATCH=False; not submitting the batch.")
    else:
        mapping = ",".join(f"{k}={v}" for k, v in json.loads(COLUMN_MAPPING_LOCAL.read_text()).items())
        sh(
            [
                "wb",
                "workflow",
                "job",
                "run",
                f"--workflow={WORKFLOW_ID}",
                f"--output-bucket-id={OUTPUT_BUCKET_ID}",
                f"--output-path={OUTPUT_PATH}",
                f"--batch-input-bucket-id={OUTPUT_BUCKET_ID}",
                f"--batch-input-csv-path={BATCH_CSV_URI_PATH}",
                f"--column-mapping={mapping}",
                "--delete-intermediate-outputs",
            ]
        )


## Monitor / cancel

Re-run this cell while a job is in flight. Set `CANCEL_JOB_ID` in the config
cell only when you intend to cancel.


In [ ]:
sh(["wb", "workflow", "job", "list", f"--workflow={WORKFLOW_ID}", "--limit=20"], check=False)

job_id = os.environ.get("LOCITYPER_JOB_ID", "").strip()
if job_id:
    sh(["wb", "workflow", "job", "describe", f"--job-id={job_id}"], check=False)
    sh(["wb", "workflow", "job", "task", "list", f"--job-id={job_id}"], check=False)

if CANCEL_JOB_ID:
    sh(["wb", "workflow", "job", "cancel", f"--job-id={CANCEL_JOB_ID}"])
else:
    print("CANCEL_JOB_ID empty; nothing cancelled.")
